# Módulo 5 — Interfaz de análisis de frotis sanguíneo
**Grupo 12 · PSIB 2026 Q2**

Aplicación Streamlit desde Google Colab. Permite **cargar una imagen de 224 × 224 píxeles**, visualizar preprocesamiento, segmentación, descriptores y resultado de clasificación; consultar las métricas reales de prueba; y descargar el resultado individual en **CSV**.

**No es una herramienta de diagnóstico médico.** Utiliza las reglas interpretables del Módulo 4, sin reentrenamiento ni cambios de parámetros.

### Archivos necesarios
- **Obligatorio:** `modulo4_reglas_finales.json` generado por la última celda del Módulo 4.
- **Recomendado:** `modulo4_predicciones_prueba.csv` generado por el Módulo 4; permite calcular y visualizar las métricas y matriz de confusión.

No requiere subir el dataset completo, las imágenes segmentadas originales ni los notebooks anteriores: el código reutiliza las funciones **originales** de los módulos 1 a 3, incluidas aquí para poder analizar imágenes nuevas.

## 1. Instalar dependencias

In [1]:
!pip -q install streamlit opencv-python-headless numpy pandas scipy scikit-image scikit-learn pillow

## 2. Cargar los archivos exportados por el Módulo 4
Ejecutá esta celda y seleccioná el JSON; también podés seleccionar el CSV de predicciones en la misma ventana. Los archivos se guardan en la carpeta actual de Colab.

In [2]:
from google.colab import files
from pathlib import Path
subidos = files.upload()
assert Path('modulo4_reglas_finales.json').exists(), 'Es necesario cargar modulo4_reglas_finales.json desde el Módulo 4.'
print('Reglas cargadas correctamente.')
print('Predicciones de prueba disponibles:', Path('modulo4_predicciones_prueba.csv').exists())

Saving modulo4_reglas_finales.json to modulo4_reglas_finales.json
Reglas cargadas correctamente.
Predicciones de prueba disponibles: False


## 3. Crear el pipeline y la aplicación
Se conservan sin cambiar las implementaciones de `preprocesar`, `segmentar_componentes`, `descriptores_forma`, `descriptores_color`, `descriptores_textura` y sus funciones auxiliares del Módulo 3. La clasificación utiliza el JSON del Módulo 4.

In [3]:
from pathlib import Path
Path("tpi_pipeline.py").write_text("# Funciones originales copiadas sin alterar la lógica desde 03_descriptores.ipynb.\n# No requiere descargar el dataset ni cargar máscaras de referencia.\nimport numpy as np\nimport pandas as pd\nimport cv2\nfrom scipy import ndimage as ndi\nfrom skimage import measure, segmentation\nfrom skimage.feature import graycomatrix, graycoprops, local_binary_pattern\n\nTAM_EE = 1 / 3\nAREA_MIN_NUC = 100\nFORMA = ['area', 'perimetro', 'circularidad', 'excentricidad', 'solidez', 'n_nucleos', 'frac_nuclear']\nCOLOR = ['H_media', 'H_desvio', 'S_media', 'S_desvio', 'a_media', 'a_desvio', 'b_media', 'b_desvio', 'S_rel']\nNIVELES = 32\nDISTANCIAS = [1, 3]\nANGULOS = [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]\nPROPS = {'contrast': 'contraste', 'homogeneity': 'homogeneidad', 'energy': 'energia', 'correlation': 'correlacion'}\nTEXTURA = [f'glcm_{v}_d{d}' for d in DISTANCIAS for v in PROPS.values()] + ['lbp_entropia', 'lbp_plano', 'lbp_borde', 'lbp_no_uniforme']\nFEATURES = FORMA + COLOR + TEXTURA\n\n\ndef estimar_campo(v):\n    n = max(v.shape)\n    ee = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(n * TAM_EE) | 1,) * 2)\n    return cv2.GaussianBlur(cv2.morphologyEx(v, cv2.MORPH_CLOSE, ee), (0, 0), n / 16)\n\ndef preprocesar(rgb):\n    h, s, v = cv2.split(cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV))\n    campo = np.maximum(estimar_campo(v.astype(np.float32)), 1)\n    v = np.clip(v / campo * campo.mean(), 0, 255).astype(np.uint8)\n    return cv2.cvtColor(cv2.merge([h, s, v]), cv2.COLOR_HSV2RGB)\n\ndef limpiar(mask, area_min=25):\n    m = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))\n    n, etq, stats, _ = cv2.connectedComponentsWithStats(m)\n    grandes = [i for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= area_min]\n    return np.isin(etq, grandes)\n\ndef sin_marca(mask):\n    # Descarta la esquina inferior derecha (marca de agua / barra de escala).\n    m = mask.copy(); h, w = m.shape\n    m[int(h * 0.88):, int(w * 0.75):] = False\n    return m\n\ndef kmeans_clusters(rgb, k=3):\n    # k-means en (a*, b*) sobre todos los píxeles. Devuelve etiquetas ordenadas por croma: 0 = fondo, ..., k-1 = núcleo.\n    cv2.setRNGSeed(42)    # semilla fija: la inicialización de k-means es aleatoria; así los resultados son reproducibles\n    ab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)[..., 1:].reshape(-1, 2) - 128\n    _, etq, centros = cv2.kmeans(ab, k, None, (cv2.TERM_CRITERIA_EPS, 20, 0.5), 3, cv2.KMEANS_PP_CENTERS)\n    orden = np.argsort(np.argsort(np.linalg.norm(centros, axis=1)))     # rango de croma de cada cluster\n    return orden[etq.ravel()].reshape(rgb.shape[:2])\n\ndef segmentar_componentes(rgb):\n    # Núcleos etiquetados (1..N) y máscara de píxeles teñidos (no fondo).\n    cl = kmeans_clusters(rgb)\n    nuc = sin_marca(ndi.binary_fill_holes(limpiar(cl == 2, AREA_MIN_NUC)))    # = segmentar_kmeans_global del Módulo 2\n    # Teñido = no es fondo por color (cluster >= 1) O es más oscuro que el fondo (Otsu sobre L*):\n    # el citoplasma es celeste muy pálido y por croma sólo puede confundirse con el fondo, pero es más oscuro.\n    L = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)[..., 0]\n    umbral_L, _ = cv2.threshold(L, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)\n    tenido = sin_marca(limpiar((cl >= 1) | (L < umbral_L))) | nuc\n    return measure.label(nuc), tenido\n\ndef descriptores_forma(nucleos):\n    # Resumen por imagen: mediana por núcleo (sin los que tocan el borde) + n_nucleos + frac_nuclear.\n    d = dict(n_nucleos=int(nucleos.max()), frac_nuclear=(nucleos > 0).mean())\n    internos = segmentation.clear_border(nucleos)\n    usar = internos if internos.max() > 0 else nucleos\n    filas = []\n    for p in measure.regionprops(usar):\n        per = p.perimeter\n        filas.append(dict(area=p.area, perimetro=per,\n                          circularidad=4 * np.pi * p.area / per**2 if per > 0 else np.nan,\n                          excentricidad=p.eccentricity, solidez=p.solidity))\n    if not filas:\n        return {**{k: np.nan for k in FORMA}, **d}\n    return {**pd.DataFrame(filas).median().to_dict(), **d}\n\ndef media_desvio_circular(h):\n    # Media y desvío circulares del tono de OpenCV (H en [0, 180) -> ángulo en [0, 2π)).\n    ang = h.astype(float) * 2 * np.pi / 180\n    c, s = np.cos(ang).mean(), np.sin(ang).mean()\n    R = np.hypot(c, s)\n    media = (np.arctan2(s, c) % (2 * np.pi)) * 180 / (2 * np.pi)\n    desvio = np.sqrt(-2 * np.log(max(R, 1e-12))) * 180 / (2 * np.pi)\n    return media, desvio\n\ndef descriptores_color(rgb, nucleos, tenido):\n    nuc = nucleos > 0\n    if not nuc.any():\n        return {k: np.nan for k in COLOR}\n    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)\n    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(float) - [0, 128, 128]\n    h_m, h_d = media_desvio_circular(hsv[..., 0][nuc])\n    s, a, b = hsv[..., 1][nuc].astype(float), lab[..., 1][nuc], lab[..., 2][nuc]\n    resto = tenido & ~nuc\n    s_resto = hsv[..., 1][resto].mean() if resto.any() else np.nan\n    return dict(H_media=h_m, H_desvio=h_d, S_media=s.mean(), S_desvio=s.std(),\n                a_media=a.mean(), a_desvio=a.std(), b_media=b.mean(), b_desvio=b.std(),\n                S_rel=s.mean() / s_resto if s_resto and s_resto > 0 else np.nan)\n\ndef gris_normalizado(rgb, nuc):\n    # Gris reescalado a [0, 1] con los percentiles 1 y 99 de los píxeles nucleares de ESTA imagen.\n    g = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(float)\n    lo, hi = np.percentile(g[nuc], [1, 99])\n    return np.clip((g - lo) / max(hi - lo, 1), 0, 1)\n\ndef glcm_nucleo(q, region):\n    # GLCM de un núcleo: niveles 1..NIVELES dentro del núcleo, 0 afuera (los pares con 0 se descartan).\n    r0, c0, r1, c1 = region.bbox\n    parche = np.where(region.image, q[r0:r1, c0:c1], 0).astype(np.uint8)\n    P = graycomatrix(parche, DISTANCIAS, ANGULOS, levels=NIVELES + 1, symmetric=True)[1:, 1:].astype(float)\n    if (P.sum(axis=(0, 1)) == 0).any():\n        return None\n    fila = {}\n    for prop, nombre in PROPS.items():\n        valores = graycoprops(P, prop)                       # [n_distancias x n_angulos]\n        with np.errstate(all='ignore'):\n            for k, d in enumerate(DISTANCIAS):\n                fila[f'glcm_{nombre}_d{d}'] = np.nanmean(valores[k])\n    return fila\n\ndef descriptores_textura(rgb, nucleos):\n    nuc = nucleos > 0\n    if not nuc.any():\n        return {k: np.nan for k in TEXTURA}\n    gn = gris_normalizado(rgb, nuc)\n    q = (gn * (NIVELES - 1)).round().astype(np.uint8) + 1        # 1..NIVELES\n    internos = segmentation.clear_border(nucleos)\n    usar = internos if internos.max() > 0 else nucleos\n    filas = [f for f in (glcm_nucleo(q, reg) for reg in measure.regionprops(usar)) if f]\n    d = pd.DataFrame(filas).median().to_dict() if filas else {k: np.nan for k in TEXTURA if k.startswith('glcm')}\n    lbp = local_binary_pattern((gn * 255).astype(np.uint8), P=8, R=1, method='uniform')\n    interior = ndi.binary_erosion(nuc)\n    codigos = lbp[interior].astype(int) if interior.any() else lbp[nuc].astype(int)\n    h = np.bincount(codigos, minlength=10) / len(codigos)                   # códigos 0..9\n    hp = h[h > 0]\n    d.update(lbp_entropia=-(hp * np.log2(hp)).sum(), lbp_plano=h[8], lbp_borde=h[3:6].sum(), lbp_no_uniforme=h[9])\n    return d\n\ndef pipeline(rgb_crudo):\n    # Imagen RGB tal como se leyó -> (máscara de núcleos, dict con todos los descriptores).\n    rgb = preprocesar(rgb_crudo)\n    nucleos, tenido = segmentar_componentes(rgb)\n    d = {**descriptores_forma(nucleos), **descriptores_color(rgb, nucleos, tenido), **descriptores_textura(rgb, nucleos)}\n    return nucleos > 0, d\n", encoding="utf-8")
Path("app.py").write_text('"""Módulo 5. Interfaz exploratoria de clasificación de frotis sanguíneos."""\nimport io\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport cv2\nimport streamlit as st\nfrom PIL import Image, UnidentifiedImageError\nfrom sklearn.metrics import confusion_matrix\nfrom tpi_pipeline import (preprocesar, segmentar_componentes, descriptores_forma,\n                          descriptores_color, descriptores_textura, FEATURES)\n\nst.set_page_config(page_title=\'Análisis de frotis | Grupo 12\', page_icon=\'🔬\', layout=\'wide\')\nBASE = Path(__file__).resolve().parent\nst.title(\'Análisis de frotis sanguíneo\')\nst.caption(\'TP de Procesamiento de Señales e Imágenes Biomédicas · Grupo 12 · Clasificación binaria basada en descriptores\')\nst.info(\'Herramienta académica de análisis de imágenes. La clasificación no constituye un diagnóstico médico y no está validada para uso clínico.\')\n\n\ndef cargar_modelo():\n    ruta = BASE / \'modulo4_reglas_finales.json\'\n    if not ruta.exists():\n        return None, f\'Falta {ruta.name}: ejecutá el Módulo 4 y subí el JSON en Colab.\'\n    try:\n        modelo = json.loads(ruta.read_text(encoding=\'utf-8\'))\n        reglas = modelo.get(\'reglas\', [])\n        if (not isinstance(reglas, list) or not reglas or len(reglas) % 2 == 0 or\n            any(r.get(\'descriptor\') not in FEATURES or\n                r.get(\'signo\') not in [-1,1] or\n                not isinstance(r.get(\'umbral\'), (float,int)) for r in reglas)):\n            return None, \'El JSON no contiene reglas válidas para el pipeline de los Módulos 1–3.\'\n        if modelo.get(\'clase_positiva\') != \'patológica\':\n            return None, \'Clase positiva incompatible con el Módulo 4.\'\n        if modelo.get(\'n_reglas\') != len(reglas):\n            return None, \'Número de reglas inconsistente en el archivo JSON.\'\n        return modelo, None\n    except (ValueError, OSError, TypeError) as exc:\n        return None, f\'No se pudo leer el modelo: {exc}\'\n\n\ndef leer_imagen(contenido):\n    with Image.open(io.BytesIO(contenido)) as im:\n        return np.array(im.convert(\'RGB\'), dtype=np.uint8)\n\n\ndef analizar(rgb):\n    # Funciones del Módulo 3 originales; se conserva su secuencia de procesamiento.\n    corregida = preprocesar(rgb)\n    nucleos, tenido = segmentar_componentes(corregida)\n    d = {**descriptores_forma(nucleos),\n         **descriptores_color(corregida, nucleos, tenido),\n         **descriptores_textura(corregida, nucleos)}\n    return corregida, nucleos, d\n\n\ndef votar(descriptores, reglas):\n    detalles = []\n    for regla in reglas:\n        nombre = regla[\'descriptor\']\n        valor = float(descriptores[nombre])\n        signo = int(regla[\'signo\'])\n        umbral = float(regla[\'umbral\'])\n        voto = int(signo * valor >= umbral)\n        detalles.append({\'Descriptor\': nombre, \'Valor\': valor,\n                         \'Condición patológica\': f\'{"≥" if signo == 1 else "≤"} {umbral/signo:.5g}\',\n                         \'Voto\': \'Patológica\' if voto else \'No patológica\',\n                         \'_voto\': voto})\n    votos = sum(t[\'_voto\'] for t in detalles)\n    pred = int(votos > len(detalles) // 2)\n    return pred, votos, pd.DataFrame(detalles)\n\n\ndef datos_prueba():\n    ruta = BASE / \'modulo4_predicciones_prueba.csv\'\n    if not ruta.exists():\n        return None, \'No se cargó modulo4_predicciones_prueba.csv. Las métricas se habilitan al subirlo en Colab.\'\n    try:\n        tabla = pd.read_csv(ruta)\n        if not {\'clase\',\'prediccion\'}.issubset(tabla.columns):\n            raise ValueError(\'Faltan las columnas clase y prediccion.\')\n        etiquetas = {\'patológica\',\'no patológica\'}\n        if not set(tabla[\'clase\'].dropna()).issubset(etiquetas) or not set(tabla[\'prediccion\'].dropna()).issubset(etiquetas):\n            raise ValueError(\'Las etiquetas no coinciden con las del Módulo 4.\')\n        if tabla[[\'clase\',\'prediccion\']].isna().any().any():\n            raise ValueError(\'Existen etiquetas vacías.\')\n        return tabla, None\n    except (OSError, ValueError) as exc:\n        return None, str(exc)\n\n\ndef evaluar_prueba(tabla):\n    y = (tabla[\'clase\'] == \'patológica\').astype(int)\n    yp = (tabla[\'prediccion\'] == \'patológica\').astype(int)\n    tn, fp, fn, tp = confusion_matrix(y, yp, labels=[0,1]).ravel()\n    return {\'Sensibilidad\': tp/(tp+fn) if tp+fn else np.nan,\n            \'Especificidad\': tn/(tn+fp) if tn+fp else np.nan,\n            \'Exactitud\': (tp+tn)/len(tabla),\n            \'Precisión positiva\': tp/(tp+fp) if tp+fp else np.nan}, (tn,fp,fn,tp)\n\nmodelo, error_modelo = cargar_modelo()\nif error_modelo:\n    st.error(error_modelo)\n    st.stop()\n\nwith st.sidebar:\n    st.header(\'Navegación\')\n    opcion = st.radio(\'Sección\', [\'Clasificación\',\'Procesamiento\',\'Descriptores\',\'Evaluación\'])\n    st.divider()\n    st.caption(f\'Modelo cargado: {len(modelo["reglas"])} reglas\')\n    st.caption(\'Procesamiento diseñado y evaluado sobre imágenes de 224 × 224 píxeles.\')\n\nsubido = st.file_uploader(\'Cargar imagen de frotis (JPG o PNG)\', type=[\'jpg\',\'jpeg\',\'png\'])\nif opcion == \'Evaluación\':\n    st.subheader(\'Desempeño sobre el conjunto de prueba\')\n    tabla, err = datos_prueba()\n    if err:\n        st.warning(err)\n    else:\n        metricas, (tn,fp,fn,tp) = evaluar_prueba(tabla)\n        cols = st.columns(4)\n        for col,(nombre,valor) in zip(cols,metricas.items()):\n            col.metric(nombre, f\'{100*valor:.1f} %\')\n        st.markdown(\'**Matriz de confusión** · Filas: clase real; columnas: clase predicha\')\n        cm = pd.DataFrame([[tn,fp],[fn,tp]], index=[\'Real: no patológica\',\'Real: patológica\'],\n                          columns=[\'Pred.: no patológica\',\'Pred.: patológica\'])\n        st.dataframe(cm, use_container_width=True)\n        st.caption(f\'{len(tabla)} imágenes de prueba. Partición por imagen (no por paciente); las métricas no corresponden a la imagen cargada.\')\n        st.markdown(\'**Interpretación:** un falso negativo es una imagen patológica clasificada como no patológica; un falso positivo es una imagen no patológica clasificada como patológica.\')\n    st.info(\'Estas métricas se calcularon a partir de las predicciones exportadas por el Módulo 4 y no cambian al cargar imágenes nuevas.\')\n    st.stop()\n\nif subido is None:\n    st.info(\'Cargá una imagen para habilitar la clasificación, la visualización y la exportación CSV.\')\n    st.stop()\n\ntry:\n    rgb = leer_imagen(subido.getvalue())\nexcept (UnidentifiedImageError, OSError, ValueError) as exc:\n    st.error(f\'No se pudo abrir la imagen: {exc}\')\n    st.stop()\n\nif rgb.shape[:2] != (224,224):\n    st.error(f\'La imagen tiene dimensiones {rgb.shape[1]} × {rgb.shape[0]} píxeles. El pipeline original se evaluó para 224 × 224. No se redimensiona automáticamente para evitar alterar los descriptores y los umbrales.\')\n    st.stop()\n\ntry:\n    corregida, nucleos, desc = analizar(rgb)\nexcept Exception as exc:\n    st.error(f\'No se pudo completar el procesamiento: {type(exc).__name__}: {exc}\')\n    st.stop()\n\nif int(nucleos.max()) == 0:\n    st.warning(\'No se detectaron núcleos; no se emite clasificación.\')\n    st.stop()\n\nno_finitos = [k for k in FEATURES if k not in desc or not np.isfinite(desc[k])]\nif no_finitos:\n    st.warning(\'No se emite clasificación porque hay descriptores inválidos: \'+\', \'.join(no_finitos))\n    st.stop()\n\npred, votos, detalle = votar(desc, modelo[\'reglas\'])\netiqueta = \'Patológica\' if pred else \'No patológica\'\n\nif opcion == \'Clasificación\':\n    st.subheader(\'Resultado del análisis\')\n    izquierda,derecha = st.columns([1,1])\n    with izquierda:\n        st.image(rgb, caption=\'Imagen original\', use_container_width=True)\n    with derecha:\n        st.metric(\'Clasificación mediante votación mayoritaria\', etiqueta)\n        st.metric(\'Votos por la clase patológica\', f\'{votos}/{len(modelo["reglas"])}\')\n        st.metric(\'Núcleos detectados\', int(nucleos.max()))\n        st.metric(\'Fracción nuclear\', f\'{100*float(desc["frac_nuclear"]):.2f} %\')\n        st.caption(\'La cantidad de votos no representa una probabilidad de enfermedad.\')\n    st.markdown(\'**Detalle de las reglas aplicadas**\')\n    st.dataframe(detalle.drop(columns=\'_voto\'), hide_index=True, use_container_width=True)\n\nelif opcion == \'Procesamiento\':\n    st.subheader(\'Etapas del procesamiento\')\n    mascara = (nucleos > 0).astype(np.uint8)\n    overlay = rgb.copy()\n    contornos,_ = cv2.findContours(mascara,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)\n    cv2.drawContours(overlay,contornos,-1,(0,255,0),1)\n    cols=st.columns(2)\n    cols[0].image(rgb, caption=\'1. Imagen original (RGB)\', use_container_width=True)\n    cols[1].image(corregida, caption=\'2. Corrección de iluminación en V\', use_container_width=True)\n    cols=st.columns(2)\n    cols[0].image((mascara*255), caption=\'3. Máscara nuclear obtenida con k-means en Lab\', clamp=True, use_container_width=True)\n    cols[1].image(overlay, caption=\'4. Contornos de núcleos sobre la imagen original\', use_container_width=True)\n    st.write(f\'Núcleos detectados: **{int(nucleos.max())}** · Fracción nuclear: **{100*desc["frac_nuclear"]:.2f} %**\')\n    st.caption(\'Los núcleos contiguos pueden identificarse como una única componente. No se utilizan máscaras de referencia del dataset.\')\n\nelif opcion == \'Descriptores\':\n    st.subheader(\'Descriptores calculados\')\n    grupos = {\'Forma\':[\'area\',\'perimetro\',\'circularidad\',\'excentricidad\',\'solidez\',\'n_nucleos\',\'frac_nuclear\'],\n              \'Color\':[\'H_media\',\'H_desvio\',\'S_media\',\'S_desvio\',\'a_media\',\'a_desvio\',\'b_media\',\'b_desvio\',\'S_rel\'],\n              \'Textura\':[k for k in FEATURES if k.startswith((\'glcm_\',\'lbp_\'))]}\n    for nombre,cols in grupos.items():\n        with st.expander(nombre,expanded=nombre==\'Forma\'):\n            st.dataframe(pd.DataFrame({\'Descriptor\':cols,\'Valor\':[desc[k] for k in cols]}),hide_index=True,use_container_width=True)\n    st.markdown(\'**Reglas finales del clasificador**\')\n    st.dataframe(detalle.drop(columns=\'_voto\'),hide_index=True,use_container_width=True)\n    st.caption(\'Regla original del Módulo 4: signo × descriptor ≥ umbral. Una mayoría de votos define la clase patológica.\')\n\nst.divider()\nst.subheader(\'Exportar resultado en CSV\')\nregistro = {\'archivo\': subido.name, \'clasificacion\': etiqueta.lower(),\n            \'votos_patologica\': votos, \'n_reglas\': len(modelo[\'reglas\'])}\nregistro.update({k: float(desc[k]) for k in FEATURES})\nfor row in detalle.to_dict(\'records\'):\n    registro[\'voto_\'+row[\'Descriptor\']] = row[\'_voto\']\ncsv = pd.DataFrame([registro]).to_csv(index=False).encode(\'utf-8-sig\')\nst.download_button(\'Descargar CSV del análisis\', data=csv, file_name=\'resultado_frotis.csv\', mime=\'text/csv\')\nst.caption(\'El CSV incluye todos los descriptores, los votos individuales y la clasificación de esta imagen.\')\n', encoding="utf-8")
print("Listo: tpi_pipeline.py y app.py")

Listo: tpi_pipeline.py y app.py


## 4. Iniciar y abrir la interfaz
Ejecutá la celda y usá el panel integrado de Streamlit debajo de ella. Si ya estaba ejecutándose, la celda detiene el proceso anterior para recargar el código.


In [5]:
import subprocess, sys, time, re, urllib.request
from pathlib import Path

# Detener ejecuciones anteriores
for nombre in ['proceso_streamlit', 'proceso_tunel']:
    try:
        globals()[nombre].terminate()
    except Exception:
        pass

log_streamlit = open('/content/streamlit_modulo5.log', 'w')
proceso_streamlit = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'app.py',
     '--server.port', '8501', '--server.address', '0.0.0.0',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=log_streamlit, stderr=subprocess.STDOUT)

for _ in range(30):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health', timeout=1) as r:
            if r.status == 200:
                break
    except Exception:
        time.sleep(1)
else:
    print(Path('/content/streamlit_modulo5.log').read_text()[-3000:])
    raise RuntimeError('No fue posible iniciar Streamlit. Revisar log.')

# Túnel público (no depende del proxy de Colab)
if not Path('cloudflared').exists():
    subprocess.run(['wget', '-q', '-O', 'cloudflared',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    subprocess.run(['chmod', '+x', 'cloudflared'], check=True)
log_tunel = open('/content/tunel.log', 'w')
proceso_tunel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
                                 stdout=log_tunel, stderr=subprocess.STDOUT)
url = None
for _ in range(30):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', Path('/content/tunel.log').read_text())
    if m:
        url = m.group(0)
        break
print('Abrí la interfaz en:', url if url else '(no se generó el link; revisar /content/tunel.log)')

Abrí la interfaz en: https://reported-snapshot-drain-arctic.trycloudflare.com


## Uso y consideraciones
1. Cargá un frotis JPG/PNG de **224 × 224 píxeles** (la resolución utilizada para desarrollar y evaluar el pipeline).
2. Consultá **Clasificación**, **Procesamiento**, **Descriptores** y **Evaluación** desde la barra lateral.
3. En la sección correspondiente descargá el **CSV** con la clasificación, los 28 descriptores y los votos de cada regla.

**Limitaciones:** no hay evaluación clínica, la partición original de ajuste/prueba se hizo por imagen, la segmentación puede fusionar núcleos en contacto y la interfaz no valida la calidad biológica de la muestra. Si no hay núcleos o descriptores válidos, no se emite clasificación.

Si el enlace deja de funcionar, reiniciá la celda 4 (y, si Colab reinició su entorno, repetí las celdas desde el comienzo).